<div style="background: linear-gradient(135deg, #1a237e, #283593); padding: 40px; border-radius: 12px; text-align: center; color: white; margin-bottom: 20px;">
  <h1 style="font-size: 2.2em; margin-bottom: 10px;">📊 Análisis de Correspondencia Múltiple (ACM)</h1>
  <h2 style="font-size: 1.3em; font-weight: 300;">Perfiles de Educación y Empleo en Colombia</h2>
  <p style="margin-top: 15px; font-size: 1em; opacity: 0.85;">Fuente: DNP — Sisbén IV · Muestra anonimizada · Corte marzo 2022</p>
</div>

---
# 1. Introducción

## ¿Qué es el Análisis de Correspondencia Múltiple (ACM)?

El **Análisis de Correspondencia Múltiple (ACM)** es una técnica estadística multivariada de reducción de la dimensionalidad diseñada para trabajar con **variables categóricas**. Es la extensión natural del Análisis de Componentes Principales (PCA) al mundo de los datos cualitativos.

El ACM transforma tablas de respuestas categoriales en un **mapa visual de baja dimensión** donde:
- Categorías con perfiles de respuesta similares aparecen **cerca** en el plano.
- Categorías con perfiles opuestos aparecen **lejos** entre sí.
- Los individuos con patrones similares se agrupan en las mismas regiones.

## ¿Qué problema resuelve?

El Sisbén contiene variables como nivel educativo, actividad laboral o posición ocupacional que son **categóricas por naturaleza**. Técnicas como PCA no pueden aplicarse directamente.

| Desafío | Solución del ACM |
|---------|------------------|
| Variables no numéricas | Transforma categorías en coordenadas numéricas |
| 9 variables difíciles de visualizar | Reduce a 2–3 dimensiones interpretables |
| ¿Qué perfiles de pobreza laboral existen? | Identifica y visualiza grupos similares |
| Brecha educativa urbano-rural | ZONA se proyecta como variable ilustrativa |

## Objetivos

1. Identificar **perfiles combinados de educación y empleo** en la población Sisbén IV.
2. Detectar **asociaciones entre privaciones educativas** y **condiciones laborales** precarias.
3. Caracterizar **tipologías de individuos** según su situación educativa y ocupacional.
4. Explorar si la dimensión **urbano/rural** estructura los perfiles encontrados.

---

## 🔑 Preguntas Guía — Sección 1

> 1. ¿Qué son las variables latentes y por qué son importantes en el ACM?
> 2. ¿Qué tipo de datos analiza el ACM y por qué no se pueden analizar directamente?
> 3. ¿Qué significa reducir la dimensionalidad en este contexto?
> 4. ¿Qué diferencia hay entre ACM y el Análisis de Correspondencias Simple?
> 5. ¿Qué se gana al transformar variables categóricas en variables numéricas?

---
# 2. Contexto de los Datos

## Descripción del dataset

El **Departamento Nacional de Planeación (DNP)** publica muestras anonimizadas del Sisbén IV con información de personas, viviendas y hogares a nivel nacional.

| Atributo | Detalle |
|----------|---------|
| Archivo | `DNP_-_Sisbén_Personas_20260317.csv` |
| Marco muestral | Base Sisbén IV — corte marzo 2022 |
| Tipo de muestreo | Probabilístico estratificado (estrato, zona, clasificación) |
| Confiabilidad | 95% — margen de error relativo 5% |
| Factor de expansión | Variable `FEX` |
| Tamaño original | ~4.46 millones de registros · 48 columnas |
| Cobertura | Nacional (excluye Toribío, Cauca — anonimización) |
| Llave persona | `COD_MPIO` + `LLAVE` + `HOGAR` + `ORDEN` |

## Variables seleccionadas

### Activas (construyen las dimensiones del ACM)

| Variable | Descripción | Bloque |
|----------|-------------|--------|
| `PER015` | Sabe leer y escribir | Educación |
| `PER016` | Actualmente estudia | Educación |
| `PER017` | Nivel educativo alcanzado (reagrupado) | Educación |
| `I1` | Privación IPM — bajo logro educativo | Educación |
| `I4` | Privación IPM — rezago escolar | Educación |
| `PER019` | Actividad principal último mes | Empleo |
| `PER020` | Posición ocupacional | Empleo |
| `I7` | Privación IPM — desempleo larga duración | Empleo |
| `I8` | Privación IPM — trabajo informal | Empleo |

### Ilustrativas (se proyectan sobre el mapa sin construir dimensiones)

| Variable | Descripción |
|----------|-------------|
| `PER001` | Sexo |
| `PER018` | Cotiza a fondo de pensiones |
| `ZONA` | Zona urbana / rural |

---

## 🔑 Preguntas Guía — Sección 2

> 1. ¿Qué tipo de variables contiene la base de datos?
> 2. ¿Por qué es importante que las variables sean categóricas en el ACM?
> 3. ¿Cómo se interpretan los individuos dentro del análisis?
> 4. ¿Qué papel juegan las categorías dentro del modelo?
> 5. ¿Qué tipo de problemas reales puede representar esta base de datos?

In [ ]:
# ── Importación de librerías ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11
})
print('✅ Librerías importadas')

---
# 3. Carga y Muestreo Estratificado

## ¿Por qué n = 3.000 y no los 4.46 millones?

El ACM es una técnica donde a partir de ~5.000 registros las coordenadas del biplot se estabilizan y no cambian significativamente. Usar millones de registros solo añade tiempo de cómputo sin mejorar la calidad del análisis.

Se usa **muestreo aleatorio estratificado por `ZONA`** para garantizar representación proporcional de zonas urbana y rural.

In [ ]:
# ── Carga del CSV (solo columnas necesarias) ─────────────────────────────────
RUTA = 'DNP_-_Sisbén_Personas_20260317.csv'

COLS = ['ZONA', 'PER001', 'PER015', 'PER016', 'PER017',
        'PER018', 'PER019', 'PER020', 'I1', 'I4', 'I7', 'I8']

print('Cargando dataset...')
df_full = pd.read_csv(RUTA, usecols=COLS, low_memory=False,
                      encoding='utf-8')   # probar 'latin-1' si falla

print(f'✅ Dataset cargado: {df_full.shape[0]:,} filas × {df_full.shape[1]} columnas')
print(f'\nDistribución ZONA (original):')
print(df_full['ZONA'].value_counts(dropna=False).to_string())

In [ ]:
# ── Muestreo estratificado por ZONA ─────────────────────────────────────────
N_MUESTRA = 3000
SEMILLA   = 42

df_validos = df_full.dropna(subset=['ZONA']).copy()
frac = N_MUESTRA / len(df_validos)

df_muestra = (
    df_validos
    .groupby('ZONA', group_keys=False)
    .apply(lambda x: x.sample(frac=frac, random_state=SEMILLA))
    .reset_index(drop=True)
)

# Ajuste fino por redondeo
if len(df_muestra) > N_MUESTRA:
    df_muestra = df_muestra.sample(n=N_MUESTRA, random_state=SEMILLA).reset_index(drop=True)
elif len(df_muestra) < N_MUESTRA:
    extra = df_validos.drop(df_muestra.index, errors='ignore') \
                      .sample(n=N_MUESTRA - len(df_muestra), random_state=SEMILLA)
    df_muestra = pd.concat([df_muestra, extra]).reset_index(drop=True)

print(f'✅ Muestra estratificada: {len(df_muestra):,} registros')
print(f'\nDistribución ZONA (muestra):')
for z, c in df_muestra['ZONA'].value_counts().items():
    print(f'  ZONA {z}: {c:>5} ({100*c/len(df_muestra):.1f}%)')

---
# 4. Metodología y Bases Teóricas

## 4.1 Preprocesamiento y Limpieza

En el Sisbén, variables como `PER017` (nivel educativo) o `PER019` (actividad) están almacenadas como **códigos numéricos** que representan categorías. Se mapean a etiquetas legibles antes del ACM.

Los indicadores IPM (`I1`, `I4`, `I7`, `I8`) son **binarios 0/1** y solo se convierten a `"Sí"`/`"No"`.

`PER017` se **reagrupa** de sus categorías originales a 4 macro-niveles para mayor parsimonia y estabilidad estadística.

---

## 🔑 Preguntas Guía — Sección 4.1

> 1. ¿Por qué es necesario transformar variables categóricas a formato numérico para el ACM?
> 2. ¿Qué métodos se pueden usar para codificar variables categóricas?
> 3. ¿Qué problemas pueden surgir si no se realiza un buen preprocesamiento?

In [ ]:
# ── Diccionarios de codificación Sisbén IV ───────────────────────────────────

MAP_PER001 = {1: 'Hombre', 2: 'Mujer'}
MAP_PER015 = {1: 'Sí lee/escribe', 2: 'No lee/escribe'}
MAP_PER016 = {1: 'Sí estudia', 2: 'No estudia'}
MAP_PER018 = {1: 'Sí cotiza pensión', 2: 'No cotiza pensión'}
MAP_ZONA   = {1: 'Urbano', 2: 'Rural'}

# PER017 — Nivel educativo → 4 macro-niveles
MAP_PER017 = {
    1: 'Nivel 1 — Sin educación',   # ninguno
    9: 'Nivel 1 — Sin educación',   # preescolar
    10:'Nivel 1 — Sin educación',   # menor sin escolaridad
    2: 'Nivel 2 — Básica',          # primaria incompleta
    3: 'Nivel 2 — Básica',          # primaria completa
    4: 'Nivel 2 — Básica',          # secundaria incompleta
    5: 'Nivel 3 — Media',           # secundaria completa
    6: 'Nivel 4 — Superior',        # técnica/tecnológica
    7: 'Nivel 4 — Superior',        # universitaria
    8: 'Nivel 4 — Superior',        # posgrado
}

# PER019 — Actividad principal último mes
MAP_PER019 = {
    1: 'Trabajando',
    2: 'Buscando trabajo',
    3: 'Estudiando',
    4: 'Oficios del hogar',
    5: 'Incapacitado permanente',
    6: 'Otra actividad',
    7: 'Rentista/pensionado',
}

# PER020 — Posición ocupacional
MAP_PER020 = {
    1: 'Empleado empresa',
    2: 'Empleado doméstico',
    3: 'Independiente',
    4: 'Patrón/empleador',
    5: 'Fam. sin pago',
    6: 'Jornalero/peón',
    7: 'Otro',
}

BIN = {0: 'No', 1: 'Sí', 0.0: 'No', 1.0: 'Sí'}

print('✅ Diccionarios definidos')

In [ ]:
# ── Aplicar transformaciones a la muestra ───────────────────────────────────
df = df_muestra.copy()

df['SEXO']        = df['PER001'].map(MAP_PER001).fillna('Otro')
df['LEER']        = df['PER015'].map(MAP_PER015).fillna('Otro')
df['ESTUDIA']     = df['PER016'].map(MAP_PER016).fillna('Otro')
df['NIV_EDU']     = df['PER017'].map(MAP_PER017).fillna('Nivel 1 — Sin educación')
df['COTIZA_PEN']  = df['PER018'].map(MAP_PER018).fillna('Otro')
df['ACTIVIDAD']   = df['PER019'].map(MAP_PER019).fillna('Otra actividad')
df['POSICION_OC'] = df['PER020'].map(MAP_PER020).fillna('Otro')
df['ZONA_ETQ']    = df['ZONA'].map(MAP_ZONA).fillna('Otro')
df['I1_EDU']      = df['I1'].map(BIN).fillna('No')
df['I4_REZ']      = df['I4'].map(BIN).fillna('No')
df['I7_DEMP']     = df['I7'].map(BIN).fillna('No')
df['I8_INF']      = df['I8'].map(BIN).fillna('No')

VARS_ACTIVAS = ['LEER','ESTUDIA','NIV_EDU','I1_EDU','I4_REZ',
                'ACTIVIDAD','POSICION_OC','I7_DEMP','I8_INF']
VARS_ILUSTR  = ['SEXO','COTIZA_PEN','ZONA_ETQ']

print('✅ Variables transformadas y etiquetadas')
df[VARS_ACTIVAS + VARS_ILUSTR].head(5)

In [ ]:
# ── Verificación de frecuencias mínimas ─────────────────────────────────────
# El ACM requiere que ninguna categoría tenga n < 15 (regla práctica)

print('=' * 62)
print(' Frecuencias por categoría (mínimo recomendado: 15)')
print('=' * 62)
alertas = []
for var in VARS_ACTIVAS:
    counts = df[var].value_counts()
    for cat, cnt in counts.items():
        flag = '  ⚠️' if cnt < 15 else ''
        if cnt < 15: alertas.append((var, cat, cnt))
        print(f'  {var:<14} | {str(cat):<36} | n={cnt:>5} ({100*cnt/len(df):5.1f}%){flag}')
    print()

if not alertas:
    print('✅ Todas las categorías tienen frecuencia suficiente')
else:
    print(f'⚠️  {len(alertas)} categoría(s) con baja frecuencia — considerar agrupar o excluir')

In [ ]:
# ── Distribución de variables activas ───────────────────────────────────────
titulos = {
    'LEER':'Sabe leer y escribir', 'ESTUDIA':'Actualmente estudia',
    'NIV_EDU':'Nivel educativo', 'I1_EDU':'Privación: bajo logro educativo',
    'I4_REZ':'Privación: rezago escolar', 'ACTIVIDAD':'Actividad principal',
    'POSICION_OC':'Posición ocupacional',
    'I7_DEMP':'Privación: desempleo largo', 'I8_INF':'Privación: trabajo informal'
}
colores = ['#1565C0','#2E7D32','#C62828','#6A1B9A','#E65100',
           '#00695C','#37474F','#AD1457','#558B2F']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, (var, color) in enumerate(zip(VARS_ACTIVAS, colores)):
    ax = axes.flatten()[i]
    counts = df[var].value_counts()
    bars = ax.barh([str(c)[:30] for c in counts.index], counts.values,
                   color=color, alpha=0.82, edgecolor='white')
    ax.set_title(titulos[var], fontweight='bold', fontsize=10)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                str(val), va='center', fontsize=8)

plt.suptitle(f'Distribución de Variables Activas — Sisbén IV (n={len(df):,})',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 4.2 Construcción de Matrices

### Matriz Indicadora $Z$

Mediante **one-hot encoding** se construye $Z$ de dimensión $n \times J$ donde $J$ es el total de categorías:

$$Z = [Z_1 | Z_2 | \cdots | Z_Q]$$

### Matriz de Burt $B = Z^T Z$

Matriz simétrica que recoge todas las **tabulaciones cruzadas** entre variables. En la diagonal: frecuencias de cada categoría. Fuera: co-ocurrencias.

### Matriz de Residuos Estandarizados

$$S = D_r^{-1/2}\,(P - r c^T)\,D_c^{-1/2}$$

- $r_{ij} > +2$: **atracción** significativa entre categorías $i$ y $j$
- $r_{ij} < -2$: **repulsión** significativa

---

## 🔑 Preguntas Guía — Sección 4.2

> 1. ¿Qué representa la Matriz de Burt?
> 2. ¿Qué son los residuos estandarizados y qué indican?
> 3. ¿Qué significa comparar valores observados con valores esperados?
> 4. ¿Qué indica un residuo positivo o negativo?

In [ ]:
# ── Matriz Indicadora y Matriz de Burt ──────────────────────────────────────
df_acm = df[VARS_ACTIVAS].copy()
Z = pd.get_dummies(df_acm, columns=VARS_ACTIVAS, dtype=int)
Z_arr = Z.values.astype(float)
B = Z_arr.T @ Z_arr

print(f'Matriz Indicadora Z: {Z.shape}  ({Z.shape[0]} individuos × {Z.shape[1]} categorías)')
print(f'Matriz de Burt  B:  {B.shape}  (simétrica: {np.allclose(B, B.T)})')

fig, ax = plt.subplots(figsize=(9, 7))
block = min(28, B.shape[0])
im = ax.imshow(B[:block, :block], cmap='YlOrRd', aspect='auto')
ax.set_title(f'Bloque Matriz de Burt (primeras {block}×{block} categorías)',
             fontweight='bold')
plt.colorbar(im, ax=ax, label='Frecuencia conjunta')
plt.tight_layout()
plt.show()

---
## 4.3 Descomposición en Valores Singulares (DVS)

El núcleo matemático del ACM es la **DVS** de la matriz de residuos estandarizados:

$$S = U \cdot D \cdot V^T$$

| Matriz | Significado |
|--------|-------------|
| $U$ ($n \times k$) | Vectores singulares izquierdos → posición de **individuos** |
| $D$ ($k \times k$) | Valores singulares $\mu_k$ en orden decreciente |
| $V$ ($J \times k$) | Vectores singulares derechos → posición de **categorías** |

Los **autovalores** $\lambda_k = \mu_k^2$ miden la inercia explicada por cada dimensión.

---

## 🔑 Preguntas Guía — Sección 4.3

> 1. ¿Qué es la DVS?
> 2. ¿Qué representan las matrices U, D y V?
> 3. ¿Por qué la DVS permite reducir la dimensionalidad?
> 4. ¿Qué relación existe entre valores singulares y autovalores?

---
## 4.4 Inercia y Selección de Dimensiones

$$I_k\% = \frac{\lambda_k}{\sum_k \lambda_k} \times 100$$

Criterios de selección:
- **Scree Plot**: buscar el codo en la curva de autovalores.
- **Umbral Kaiser**: retener dimensiones con $\lambda_k > 1/Q$.
- **Inercia acumulada**: retener hasta ~70–80%.

---

## 🔑 Preguntas Guía — Sección 4.4

> 1. ¿Qué es la inercia y cómo se interpreta?
> 2. ¿Qué indica un autovalor alto?
> 3. ¿Cómo se decide cuántas dimensiones conservar?
> 4. ¿Qué diferencia hay entre coordenadas estándar y principales?
> 5. ¿Qué significa que una dimensión sea más importante que otra?

In [ ]:
# ── ACM con Prince ───────────────────────────────────────────────────────────
import prince

mca = prince.MCA(n_components=6, n_iter=10, random_state=42)
mca = mca.fit(df_acm)

eigenvalues  = mca.eigenvalues_
inercia_pct  = 100 * eigenvalues / eigenvalues.sum()
inercia_acum = np.cumsum(inercia_pct)
Q            = len(VARS_ACTIVAS)
umbral_k     = 1 / Q

print('=' * 55)
print(' Tabla de autovalores e inercia')
print('=' * 55)
print(f'{"Dim":>4} {"λ":>14} {"Inercia %":>11} {"Acumulada %":>13}')
print('-' * 48)
for k, (ev, pct, ac) in enumerate(zip(eigenvalues, inercia_pct, inercia_acum)):
    m = ' ← Kaiser' if ev > umbral_k else ''
    print(f'{k+1:>4} {ev:>14.5f} {pct:>10.2f}% {ac:>12.2f}%{m}')
print(f'\nUmbral Kaiser (1/{Q}): {umbral_k:.4f}')
print(f'Inercia acumulada Dim1+Dim2: {inercia_acum[1]:.2f}%')

In [ ]:
# ── Scree Plot ───────────────────────────────────────────────────────────────
dims = np.arange(1, len(eigenvalues)+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(dims, inercia_pct, color='#1565C0', alpha=0.75, edgecolor='white', label='Inercia por dim.')
ax1.plot(dims, inercia_acum, 'o-', color='#C62828', lw=2, ms=6, label='Acumulada')
ax1.axhline(70, color='#2E7D32', ls='--', lw=1.5, label='70%')
for i, (p, a) in enumerate(zip(inercia_pct, inercia_acum)):
    ax1.text(i+1, p+0.5, f'{p:.1f}%', ha='center', fontsize=8)
ax1.set_xticks(dims)
ax1.set_xlabel('Dimensión'); ax1.set_ylabel('% Inercia')
ax1.set_title('Scree Plot', fontweight='bold'); ax1.legend(fontsize=9)

colb = ['#1565C0' if ev > umbral_k else '#90A4AE' for ev in eigenvalues]
ax2.bar(dims, eigenvalues, color=colb, edgecolor='white')
ax2.axhline(umbral_k, color='#E65100', ls='--', lw=2, label=f'Kaiser={umbral_k:.3f}')
ax2.set_xticks(dims)
ax2.set_xlabel('Dimensión'); ax2.set_ylabel('Autovalor (λ)')
ax2.set_title('Autovalores vs Kaiser', fontweight='bold'); ax2.legend(fontsize=9)

plt.suptitle('Selección del Número Óptimo de Dimensiones — Sisbén IV',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 4.5 Visualización e Interpretación: Biplot

| Observación geométrica | Interpretación |
|------------------------|----------------|
| Dos categorías **muy cercanas** | Tienden a aparecer en los mismos individuos |
| Categorías en **extremos opuestos** | Definen el contraste del eje |
| Punto **cerca del origen** | Comportamiento promedio, sin rasgo distintivo |
| Punto **lejos del origen** | Perfil muy atípico |

### Principio del centroide
- Individuo = promedio ponderado de sus categorías.
- Categoría = promedio ponderado de los individuos que la responden.

---

## 🔑 Preguntas Guía — Sección 4.5

> 1. ¿Cómo se interpretan las distancias entre puntos en el plano factorial?
> 2. ¿Qué significa que dos categorías estén cerca?
> 3. ¿Qué representa el origen?
> 4. ¿Qué es un biplot y qué información proporciona?
> 5. ¿Qué significa la polaridad de un eje?
> 6. ¿Qué indica que un punto esté lejos del origen?

In [ ]:
# ── Coordenadas del biplot ───────────────────────────────────────────────────
cat_coords = mca.column_coordinates(df_acm).copy()
ind_coords = mca.row_coordinates(df_acm).copy()

cat_var = []
for col in cat_coords.index:
    for v in VARS_ACTIVAS:
        if col.startswith(v + '_'):
            cat_var.append(v); break
    else:
        cat_var.append('Otro')
cat_coords['Variable'] = cat_var

pal = plt.cm.get_cmap('tab10', len(VARS_ACTIVAS))
cmap_v = {v: pal(i) for i, v in enumerate(VARS_ACTIVAS)}
print(f'✅ Coordenadas: {cat_coords.shape[0]} categorías · {ind_coords.shape[0]} individuos')

In [ ]:
# ── Biplot principal ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 11))
ax.axhline(0, color='gray', lw=0.7, alpha=0.4)
ax.axvline(0, color='gray', lw=0.7, alpha=0.4)

ax.scatter(ind_coords.iloc[:,0], ind_coords.iloc[:,1],
           color='#CFD8DC', alpha=0.2, s=16, zorder=1, label='Individuos')

for var in VARS_ACTIVAS:
    sub = cat_coords[cat_coords['Variable'] == var]
    c = cmap_v[var]
    ax.scatter(sub.iloc[:,0], sub.iloc[:,1],
               color=c, s=100, zorder=4, edgecolors='white', lw=0.6)
    for idx, row in sub.iterrows():
        lbl = str(idx).replace(var+'_','')[:32]
        ax.annotate(lbl, xy=(row.iloc[0], row.iloc[1]),
                    xytext=(6,4), textcoords='offset points',
                    fontsize=7.5, color=c, fontweight='medium')

handles = [mpatches.Patch(color=cmap_v[v], label=v) for v in VARS_ACTIVAS]
handles.append(plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor='#CFD8DC', ms=8, label='Individuos'))
ax.legend(handles=handles, loc='upper right', fontsize=8.5,
          framealpha=0.92, title='Variables')

ax.set_xlabel(f'Dimensión 1  ({inercia_pct[0]:.2f}% de inercia)', fontsize=12)
ax.set_ylabel(f'Dimensión 2  ({inercia_pct[1]:.2f}% de inercia)', fontsize=12)
ax.set_title(
    f'Mapa Factorial ACM — Perfiles Educación y Empleo\n'
    f'Sisbén IV · Muestra estratificada n={len(df):,} · Corte marzo 2022',
    fontsize=13, fontweight='bold', pad=14)
plt.tight_layout(); plt.show()

In [ ]:
# ── Proyección de variables ilustrativas ─────────────────────────────────────
# Se calcula como el centroide (promedio) de coordenadas de individuos
# que pertenecen a cada categoría ilustrativa.

MAP_COL = {'SEXO':'SEXO', 'COTIZA_PEN':'COTIZA_PEN', 'ZONA_ETQ':'ZONA_ETQ'}
MRK = {'SEXO':'D', 'COTIZA_PEN':'s', 'ZONA_ETQ':'^'}
COL_ILUS = {'SEXO':'#AD1457', 'COTIZA_PEN':'#00695C', 'ZONA_ETQ':'#E65100'}

fig, ax = plt.subplots(figsize=(14, 10))
ax.axhline(0, color='gray', lw=0.6, alpha=0.4)
ax.axvline(0, color='gray', lw=0.6, alpha=0.4)

# Individuos de fondo
ax.scatter(ind_coords.iloc[:,0], ind_coords.iloc[:,1],
           color='#ECEFF1', alpha=0.18, s=14, zorder=1)

# Categorías activas (pequeñas)
for var in VARS_ACTIVAS:
    sub = cat_coords[cat_coords['Variable']==var]
    ax.scatter(sub.iloc[:,0], sub.iloc[:,1], color=cmap_v[var], s=50, alpha=0.55, zorder=3)

# Variables ilustrativas
for v_ilus in VARS_ILUSTR:
    col = MAP_COL[v_ilus]
    for cat in df[col].unique():
        mask = df[col].values == cat
        x = ind_coords.iloc[mask, 0].mean()
        y = ind_coords.iloc[mask, 1].mean()
        ax.scatter(x, y, marker=MRK[v_ilus], color=COL_ILUS[v_ilus],
                   s=210, zorder=6, edgecolors='white', lw=1.2)
        ax.annotate(f'{v_ilus}_{cat}', xy=(x,y), xytext=(8,6),
                    textcoords='offset points', fontsize=9,
                    fontweight='bold', color=COL_ILUS[v_ilus])

hl = [
    plt.Line2D([0],[0],marker='D',color='w',markerfacecolor='#AD1457',ms=10,label='SEXO (ilustrativa)'),
    plt.Line2D([0],[0],marker='s',color='w',markerfacecolor='#00695C',ms=10,label='COTIZA_PEN (ilustrativa)'),
    plt.Line2D([0],[0],marker='^',color='w',markerfacecolor='#E65100',ms=10,label='ZONA (ilustrativa)'),
]
ax.legend(handles=hl, loc='lower right', fontsize=9, framealpha=0.92)
ax.set_xlabel(f'Dimensión 1 ({inercia_pct[0]:.2f}%)', fontsize=12)
ax.set_ylabel(f'Dimensión 2 ({inercia_pct[1]:.2f}%)', fontsize=12)
ax.set_title('Biplot con Variables Ilustrativas\n(Sexo · Pensión · Zona urbano/rural)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

---
# 5. Resultados

## Interpretación de los ejes

Para interpretar cada dimensión se identifican las categorías en sus extremos:
- **Polo positivo (+)**: categorías con coordenadas más altas → definen un perfil.
- **Polo negativo (−)**: categorías con coordenadas más bajas → definen el perfil contrastante.

---

## 🔑 Preguntas Guía — Sección 5

> 1. ¿Qué patrones o asociaciones se identificaron?
> 2. ¿Qué dimensiones resultaron más relevantes y por qué?
> 3. ¿Qué perfiles de individuos se pueden identificar?
> 4. ¿Qué conclusiones se pueden extraer del análisis?
> 5. ¿Qué limitaciones tiene el modelo?
> 6. ¿Se logró una representación parsimoniosa?

In [ ]:
# ── Polaridades por dimensión ────────────────────────────────────────────────
cat_num = cat_coords.iloc[:,:2].copy()
cat_num.columns = ['Dim1','Dim2']
cat_num['Variable'] = cat_var

for dim in ['Dim1','Dim2']:
    print(f'\n{"="*60}')
    print(f' DIMENSIÓN {dim[-1]} — Top 5 por polo')
    print(f'{"="*60}')
    for label, fn in [("Polo (+)", 'nlargest'), ("Polo (−)", 'nsmallest')]:
        top = getattr(cat_num, fn)(5, dim)[[dim,'Variable']]
        print(f'\n  {label}:')
        for idx, row in top.iterrows():
            lbl = str(idx).replace(row['Variable']+'_','')[:40]
            print(f'    {row[dim]:+.4f}  [{row["Variable"]}]  {lbl}')

In [ ]:
# ── Contribución de variables ────────────────────────────────────────────────
contrib_d1 = cat_num.groupby('Variable')['Dim1'].apply(lambda x:(x**2).mean())
contrib_d2 = cat_num.groupby('Variable')['Dim2'].apply(lambda x:(x**2).mean())
df_c = pd.DataFrame({'Dimensión 1':contrib_d1, 'Dimensión 2':contrib_d2})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (dim, color) in enumerate([('Dimensión 1','#1565C0'),('Dimensión 2','#6A1B9A')]):
    vals = df_c[dim].sort_values()
    bars = axes[i].barh(vals.index, vals.values, color=color, alpha=0.78, edgecolor='white')
    axes[i].set_title(f'Contribución por variable\n{dim}', fontweight='bold')
    axes[i].set_xlabel('Inercia media (coord²)')
    for bar, val in zip(bars, vals.values):
        axes[i].text(bar.get_width()+0.0003, bar.get_y()+bar.get_height()/2,
                     f'{val:.4f}', va='center', fontsize=8.5)
plt.suptitle('¿Qué variables definen cada dimensión? — Sisbén IV',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Perfiles de individuos por cuadrante ─────────────────────────────────────
ind_df = ind_coords.iloc[:,:2].copy()
ind_df.columns = ['Dim1','Dim2']
ind_df['ZONA']     = df['ZONA_ETQ'].values
ind_df['NIV_EDU']  = df['NIV_EDU'].values
ind_df['ACTIVIDAD']= df['ACTIVIDAD'].values

ind_df['Cuadrante'] = 'Q3 (−,−)'
ind_df.loc[(ind_df.Dim1>=0)&(ind_df.Dim2>=0),'Cuadrante'] = 'Q1 (+,+)'
ind_df.loc[(ind_df.Dim1< 0)&(ind_df.Dim2>=0),'Cuadrante'] = 'Q2 (−,+)'
ind_df.loc[(ind_df.Dim1>=0)&(ind_df.Dim2< 0),'Cuadrante'] = 'Q4 (+,−)'

print('=== Distribución por cuadrante ===')
print(ind_df['Cuadrante'].value_counts().to_string())

color_zona = {'Urbano':'#1565C0','Rural':'#E65100','Otro':'#888780'}
fig, ax = plt.subplots(figsize=(11, 8))
ax.axhline(0, color='gray', lw=0.6, alpha=0.4)
ax.axvline(0, color='gray', lw=0.6, alpha=0.4)

for z, grp in ind_df.groupby('ZONA'):
    c = color_zona.get(str(z),'#888780')
    ax.scatter(grp.Dim1, grp.Dim2, label=f'ZONA: {z}',
               color=c, s=22, alpha=0.5, edgecolors='none')

for txt, (px,py) in [('Q1(+,+)',(0.97,0.97)),('Q2(−,+)',(0.03,0.97)),
                      ('Q3(−,−)',(0.03,0.03)),('Q4(+,−)',(0.97,0.03))]:
    ax.text(px, py, txt, transform=ax.transAxes,
            ha='right' if px>0.5 else 'left',
            va='top' if py>0.5 else 'bottom',
            fontsize=9, color='#546E7A', style='italic')

ax.legend(fontsize=10, framealpha=0.9)
ax.set_xlabel(f'Dimensión 1 ({inercia_pct[0]:.1f}%)', fontsize=11)
ax.set_ylabel(f'Dimensión 2 ({inercia_pct[1]:.1f}%)', fontsize=11)
ax.set_title(f'Perfiles de individuos — Sisbén IV (n={len(df):,})\ncoloreado por ZONA urbano/rural',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Panel resumen ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

ax1 = fig.add_subplot(gs[0,0])
ax1.bar(dims, inercia_pct, color='#1565C0', alpha=0.75, edgecolor='white')
ax1.plot(dims, inercia_acum, 'o-', color='#C62828', lw=2, ms=5)
ax1.axhline(70, color='#2E7D32', ls='--', lw=1.5)
ax1.set_title('Scree Plot', fontweight='bold')
ax1.set_xlabel('Dimensión'); ax1.set_ylabel('% Inercia'); ax1.set_xticks(dims)

ax2 = fig.add_subplot(gs[0,1])
ax2.axhline(0, color='gray', lw=0.6, alpha=0.4)
ax2.axvline(0, color='gray', lw=0.6, alpha=0.4)
for var in VARS_ACTIVAS:
    sub = cat_coords[cat_coords['Variable']==var]
    ax2.scatter(sub.iloc[:,0], sub.iloc[:,1], color=cmap_v[var], s=55, alpha=0.85)
ax2.set_title('Mapa de categorías', fontweight='bold')
ax2.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}%)')
ax2.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}%)')

ax3 = fig.add_subplot(gs[1,0])
v1 = df_c['Dimensión 1'].sort_values()
ax3.barh(v1.index, v1.values, color='#1565C0', alpha=0.78, edgecolor='white')
ax3.set_title('Contribución por variable — Dim 1', fontweight='bold')
ax3.set_xlabel('Inercia media')

ax4 = fig.add_subplot(gs[1,1])
qc = ind_df['Cuadrante'].value_counts()
ax4.bar(qc.index, qc.values, color=['#1565C0','#2E7D32','#C62828','#E65100'],
        alpha=0.8, edgecolor='white')
ax4.set_title('Individuos por cuadrante', fontweight='bold')
ax4.set_ylabel('N')
for bar, val in zip(ax4.patches, qc.values):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
             str(val), ha='center', fontweight='bold')

fig.suptitle(f'Panel Resumen ACM — Educación y Empleo · Sisbén IV (n={len(df):,})',
             fontsize=15, fontweight='bold', y=1.01)
plt.show()
print('\n✅ Análisis completado.')

---
# 6. Conclusiones

## Síntesis del análisis

El ACM aplicado al Sisbén IV permite identificar los principales patrones de asociación entre educación y empleo en la población registrada.

## Limitaciones

| Limitación | Descripción |
|------------|-------------|
| Muestra n=3.000 | Representativa pero no sustituye el análisis poblacional completo |
| Sobreestimación de inercia | El ACM clásico subestima la inercia real (corrección de Benzécri disponible) |
| Interpretación subjetiva | El nombre de las dimensiones lo asigna el analista |
| Corte temporal | Datos del corte marzo 2022 — no reflejan cambios recientes |
| Codificación | Los mapas `PER017`/`PER019`/`PER020` se basan en la documentación pública del Sisbén IV — verificar contra el diccionario oficial antes de publicar resultados |

## Flujo completo

```
CSV Sisbén IV (4.46M registros)
        ↓
Muestreo estratificado por ZONA → n = 3.000
        ↓
Codificación: mapas Sisbén IV + reagrupación PER017
        ↓
Matriz Indicadora Z  →  Matriz de Burt B = ZᵀZ
        ↓
Matriz residuos S = Dr⁻¹/²(P − rcᵀ)Dc⁻¹/²
        ↓
DVS: S = UDVᵀ  →  λₖ = μₖ²  →  Scree Plot
        ↓
Biplot + proyección ilustrativas (ZONA, SEXO, PENSIÓN)
        ↓
Interpretación de perfiles educativos/laborales
```

## Referencias

- Greenacre, M. (2017). *Correspondence Analysis in Practice* (3rd ed.). CRC Press.
- DNP (2022). *Documentación técnica Sisbén IV*. Departamento Nacional de Planeación.
- Datos Abiertos Colombia: https://www.datos.gov.co/Estad-sticas-Nacionales/DNP-Sisb-n-Personas/hq2v-5umk
- Abdi, H. & Valentin, D. (2007). Multiple Correspondence Analysis. *Encyclopedia of Measurement and Statistics*.